In [1]:
### Parsing and Loading PDFs 

In [9]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
)

In [6]:
try:
    pdf_loader = PyPDFLoader("data/pdf/attention.pdf")
    pdf_documents = pdf_loader.load()
    print(f"pdf_loader - Total documents loaded: {len(pdf_documents)}")
    print(f"pdf_loader - First document metadata: {pdf_documents[0].metadata}")
    print(f"pdf_loader - First document content preview: {pdf_documents[0].page_content[:100]}")
except Exception as e:
    print(f"Error loading PDF with PyPDFLoader: {e}")    

pdf_loader - Total documents loaded: 1
pdf_loader - First document metadata: {'producer': 'Microsoft® Excel® for Microsoft 365', 'creator': 'Microsoft® Excel® for Microsoft 365', 'creationdate': '2025-12-16T15:31:08+05:30', 'author': 'Ashwin Shukla/HR/MUM CORP/MH', 'moddate': '2025-12-16T15:31:08+05:30', 'source': 'data/pdf/attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}
pdf_loader - First document content preview: Dates Day Holidays Type of Holiday North East South West & 
Corporate
01-Jan-26 Thursday New Years D


In [11]:
try: 
    mupdf_loader = PyMuPDFLoader("data/pdf/attention.pdf")
    mupdf_documents = mupdf_loader.load()
    print(f"mupdf_loader - Total documents loaded: {len(mupdf_documents)}")
    print(f"mupdf_loader - First document metadata: {mupdf_documents[0].metadata}")
    print(f"mupdf_loader - First document content preview: {mupdf_documents[0].page_content[:100]}")
except Exception as e:
    print(f"Error loading PDF with PyMuPDFLoader: {e}")    

mupdf_loader - Total documents loaded: 1
mupdf_loader - First document metadata: {'producer': 'Microsoft® Excel® for Microsoft 365', 'creator': 'Microsoft® Excel® for Microsoft 365', 'creationdate': '2025-12-16T15:31:08+05:30', 'source': 'data/pdf/attention.pdf', 'file_path': 'data/pdf/attention.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Ashwin Shukla/HR/MUM CORP/MH', 'subject': '', 'keywords': '', 'moddate': '2025-12-16T15:31:08+05:30', 'trapped': '', 'modDate': "D:20251216153108+05'30'", 'creationDate': "D:20251216153108+05'30'", 'page': 0}
mupdf_loader - First document content preview: Dates
Day
Holidays
Type of Holiday
North
East
South
West & 
Corporate
01-Jan-26
Thursday
New Years D


In [19]:
raw_pdf_text = """Company Financial Report


    The ﬁnancial performance for ﬁscal year 2024
    shows signiﬁcant growth in proﬁtability.
    
    
    
    Revenue increased by 25%.
    
The company's efﬁciency improved due to workﬂow
optimization.


Page 1 of 10
"""

def clean_text(text):
    cleaned_text = " ".join(text.split())
    cleaned_text.replace('ﬁ', "fi")
    cleaned_text.replace('ﬂ', "fl")
    return cleaned_text

print(clean_text(raw_pdf_text))



Company Financial Report The ﬁnancial performance for ﬁscal year 2024 shows signiﬁcant growth in proﬁtability. Revenue increased by 25%. The company's efﬁciency improved due to workﬂow optimization. Page 1 of 10


In [23]:
from langchain_core.documents import Document
from typing import List
from langchain_community.document_loaders import (
    PyMuPDFLoader
)
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)
class SmartPDFProcessor:
    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            separators = ["\n\n", "\n", " ", ""],
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            length_function = len,
        )

    def process_pdf(self, path:str) -> List[Document]:
        loaded = PyMuPDFLoader(path)
        final = loaded.load()
        print(final)
        processed_chunks = []
        for i, doc in enumerate(final):
            print(f" page content error {doc.page_content}")
            cleaned = self._clean_text(doc.page_content)

            if(len(cleaned.strip()) < 50):
                continue
            chunks = self.text_splitter.create_documents(
                texts=[cleaned],
                metadatas=[{
                    **doc.metadata,
                    "page": i + 1,
                    "total_pages": len(final),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned)
                }]
            )
            processed_chunks.extend(chunks)
            return processed_chunks
    def _clean_text(self, text):
        cleaned_text = " ".join(text.split())
        cleaned_text.replace('ﬁ', "fi")
        cleaned_text.replace('ﬂ', "fl")
        return cleaned_text
pdf_processor = SmartPDFProcessor()
try:
    smart_chunks=pdf_processor.process_pdf("data/pdf/attention.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")
    for key, value in smart_chunks[0].metadata.items():
        print(f"{key}: {value}")
except Exception as e:
    print(f"Processing error: {e}")    

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'data/pdf/attention.pdf', 'file_path': 'data/pdf/attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\ni

In [21]:
pdf_loader = PyMuPDFLoader("data/pdf/attention.pdf").load()
print(pdf_loader)
processed_chunks = []
print(f"pdf_loader - Total documents loaded: {len(pdf_loader)}")
for index, doc in enumerate(pdf_loader):
    # print(f"Document page content {index+1} {doc.page_content[:100]}")
    # print(f"document metadata {index+1} {doc.metadata}")
    cleaned_doc  = clean_text(doc.page_content) 
    chunks = RecursiveCharacterTextSplitter(
            separators = ["\n\n", "\n", " ", ""],
            chunk_size = 1000,
            chunk_overlap = 100,
            length_function = len,
        )
    chunks_doc = chunks.create_documents(texts=[cleaned_doc], metadatas= [{
                    **doc.metadata,
                    "page": index + 1,
                    "total_pages": len(pdf_loader),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(doc.page_content)
                }]
                )
    print(f"length of chunks_docs {len(chunks_doc)}")
    for i, chunk in enumerate(chunks_doc):

        print('------------------------')
        print(f"page content of chunk {i+1} {chunk.page_content[:100]}")
        print(f"metadata for chunk{chunk.metadata}")


[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'data/pdf/attention.pdf', 'file_path': 'data/pdf/attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\ni